In [4]:
import duckdb
from pathlib import Path

con = duckdb.connect()

mapping_path = Path("../../data/external/cik_to_permno.csv.gz")
cik_path = Path("../../data/pulled/cik_list_all.csv")
matched_out = Path("../../data/pulled/cik_permno_matched.csv")

con.execute(f"""
COPY (
    SELECT DISTINCT
        m.cik,
        m.LPERMNO AS permno,
        m.LPERMCO AS permco,
        m.GVKEY AS gvkey,
        m.LINKTYPE AS linktype,
        m.LINKDT AS linkdt,
        m.LINKENDDT AS linkenddt,
        m.tic,
        m.conm
    FROM read_csv_auto(
        '{mapping_path}',
        all_varchar = true,
        ignore_errors = true
    ) AS m
    INNER JOIN read_csv_auto(
        '{cik_path}',
        all_varchar = true
    ) AS c
        ON TRY_CAST(m.cik AS BIGINT) = TRY_CAST(c.cik AS BIGINT)
    WHERE m.cik IS NOT NULL
      AND m.LPERMNO IS NOT NULL
      AND TRY_CAST(m.cik AS BIGINT) IS NOT NULL
      AND TRY_CAST(c.cik AS BIGINT) IS NOT NULL
) TO '{matched_out}' WITH (HEADER, DELIMITER ',');
""")

print("Saved to:", matched_out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved to: ../../data/pulled/cik_permno_matched.csv
